# Exploring the recruit data

Charts of ratings by region and school, built on `data/recruits.csv`. Run `python clean.py` first if the CSV is missing.

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


In [ ]:
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)

In [ ]:
# recruits.csv is built by clean.py from the raw files in data/raw/
df = pd.read_csv("../data/recruits.csv")
df.head()

In [ ]:
# Convert region to numeric (non-numeric becomes NaN)
df['region'] = pd.to_numeric(df['region'], errors='coerce')

# Keep only rows where region is 1 through 8
df = df[df['region'].between(1, 8)]

In [ ]:
# Matplotlib histogram per region
regions = sorted(df['region'].dropna().unique())
fig, axes = plt.subplots(len(regions), 1, figsize=(8, 2.2*len(regions)), sharex=True)
if len(regions) == 1: axes = [axes]
for ax, r in zip(axes, regions):
    vals = df.loc[df['region']==r, 'rating'].dropna()
    ax.hist(vals, bins=20)
    ax.set_title(f"Rating distribution — {r}")
    ax.set_ylabel("Count")
axes[-1].set_xlabel("Rating")
plt.tight_layout()
plt.show()

# Seaborn box + jitter
plt.figure(figsize=(10,5))
sns.boxplot(data=df, x='region', y='rating')
sns.stripplot(data=df, x='region', y='rating', alpha=0.35, size=3, color='k')
plt.title("Rating by Region")
plt.tight_layout(); plt.show()

# Plotly violin (interactive)
fig = px.violin(df, x="region", y="rating", box=True, points="all", title="Rating by Region")
fig.show()


In [ ]:
for c in ['region', 'level', 'school_commit']:
    if c in df.columns:
        df[c] = df[c].astype(str).str.strip()

In [ ]:
# Keep only rows that have a school (committed or target)
commits = df[df['school'].notna() & (df['school'] != '')].copy()

# Aggregates: size & strength of each class
agg = (commits
       .groupby(['school','grad_year'])
       .agg(n_commits=('athlete','count'),
            avg_rating=('rating','mean'),
            med_rating=('rating','median'))
       .reset_index()
       .sort_values(['school','grad_year']))
agg.head()

# Stacked bar: class size per school by year
counts = (commits
          .groupby(['school','grad_year'])['athlete']
          .count()
          .unstack(fill_value=0))
counts.plot(kind='bar', stacked=True, figsize=(10,6))
plt.title("Recruiting Class Size by Year")
plt.xlabel("School"); plt.ylabel("# commits")
plt.tight_layout(); plt.show()

# Composition by level (stacked bars)
lvl_counts = (commits
              .groupby(['school','level'])['athlete']
              .count()
              .reset_index())
fig = px.bar(lvl_counts, x='school', y='athlete', color='level',
             title="Commit Composition by Level")
fig.show()

# Trend: average rating per school across grad_year
avg_by_year = (commits
               .groupby(['school','grad_year'])['rating']
               .mean()
               .reset_index())
fig = px.line(avg_by_year, x='grad_year', y='rating', color='school',
              markers=True, title="Avg Rating by Class Year (per School)")
fig.show()


In [ ]:
def top_prospects(data, level=None, grad_year=None, k=10, by='rating'):
    q = data.copy()
    if level is not None:
        q = q[q['level'].str.lower() == str(level).lower()]
    if grad_year is not None:
        q = q[q['grad_year'] == grad_year]
    return (q
            .sort_values(by=by, ascending=False)
            .loc[:, ['athlete','level','grad_year','region','state','country','rating',
                     'vt_high','ub_high','bb_high','fx_high','aa_high','school','is_elite','has_perfect10']]
            .head(k))

display(top_prospects(df, level="Elite", grad_year=2027, k=10))
display(top_prospects(df, level="Level 10", grad_year=2026, k=10))
display(top_prospects(df, grad_year=2025, k=10))


In [ ]:
top_aa = df.nlargest(10, 'aa_high')[['athlete', 'aa_high']]

plt.figure(figsize=(10,6))
plt.barh(top_aa['athlete'], top_aa['aa_high'])
plt.xlabel('All-Around Score')
plt.title('Top 10 Recruits by All-Around')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
import plotly.express as px

fig = px.histogram(df, x="region", title="Distribution of Recruits by Region")
fig.show()